## Exploratory data anaylsis


In [ ]:
import os
import sys
import gc
import cv2
import gzip

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import squidpy as sq

import torch
import tifffile
import xml.etree.ElementTree as ET

from collections import OrderedDict
from skimage.transform import rescale
from IPython.display import display

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.font_manager
from matplotlib import rcParams

rcParams.update({'font.size': 10})
rcParams.update({'figure.dpi': 300})
rcParams.update({'savefig.dpi': 300})

import warnings
warnings.filterwarnings('ignore')

In [ ]:
sys.path.append('../')
sys.path.append('../util')
import utils, gen_graph

### CyCIF multiplexed image registration

#### Multiplexed image registration (full 3D sample)

| Cycle | Opal 520 | Opal 690 | Opal 570 | Opal 780 | 
| --- | --- | --- | --- | --- |
| 1 | B-Catenin | ASS1 | GS | CYP3A4 |
| 2 | Pan-CK | CD31 | Collagen | NaN |
| 3 | CD45 | CD68 | Arg1 | Lyve1 |
| 4 | CD56 | Vimentin | PU1 | CD3 |

(1). Load Cy-IF images from google cloud bucket

In [ ]:
CREDENTIAL_PATH = "../data/azizilab-cell-segmentation-05f1a1125db2.json"
PROJECT_ID = 'azizilab-cell-segmentation'
BUCKET_ID = 'liver_3d'
HOME_PATH = 'CyIF/qupath/'

cyif_reader = IO.CyIFGcloudReader(CREDENTIAL_PATH,
                                  PROJECT_ID,
                                  BUCKET_ID,
                                  HOME_PATH,
                                  scale=1,
                                  sigma=10)


(2). Within-cycle registration

In [ ]:
out_path = '../data/cycif/raw/'

for slide_id in cyif_reader.slide_ids:
    annot_imgs = cyif_reader.load_imgs(slide_id,
                                       chans_to_ignore={'Opal 620'},
                                       verbose=True)    
    
    annot_imgs_warped = cyif_reader.register_cycles(annot_imgs, 
                                                    verbose=True)
    
    cyif_reader.save_annotated_imgs(annot_imgs_warped, 
                                    output_path=out_path)

    gc.collect()
    # del annot_imgs, annot_imgs_warped

#### 2D/3D zonation

- TODO: **Figure out misalignment issues**, currently only works for Slide 1 & 3
- Selected zonation markers: `GS 647`, `CYP3A4`, `ASS1 PE` (Round 1), `Col I`, `Pan CK` (Round 2)
- Try construct 3D graphs

In [ ]:
from skimage.filters import gaussian, sobel, threshold_otsu, threshold_minimum
from skimage.exposure import rescale_intensity
from skimage.registration import optical_flow_ilk
from scipy import ndimage as ndi

In [ ]:
from importlib import reload
reload(IO)

Load aligned CyIF tiffs:

In [ ]:
data_path = '../data/cycif/aligned/registered_slides/'
cyif_imgs, filenames = IO.load_annot_tiffs(data_path, ext='ome.tiff')

# TODO: rerun on full z-slices after solving the misalignment issues
cyif_imgs, filenames = cyif_imgs[:7], filenames[:7]
filenames

In [ ]:
chan_labels = cyif_imgs[0].keys()
print(list(chan_labels))

Examine continuity of zonation markers:

In [ ]:
# Compute ROI (foreground) mask  + AF mask for each z-slice
# roi_masks = [utils.get_roi_mask(img['DAPI'])
#              for img in cyif_imgs]

af1_masks = [(img['Sample AF_01'] > 40).astype(np.uint8)
             for img in cyif_imgs]
af2_masks = [(img['Sample AF_02'] > 40).astype(np.uint8)
             for img in cyif_imgs]

In [ ]:
# Try subtracting w/ AF within ROI foregrond regions
# Normalize each channel to [0, 1]

zone_labels = ['GS 647', 'CYP3A4', 'ASS1 PE', 'Col I', 'Pan CK']
zonation_chans = {}

for label in zone_labels:
    chans = []
    for i, (img, roi) in enumerate(zip(cyif_imgs, roi_masks)):
        af = af1_masks[i] if label in zone_labels[:3] else af2_masks[i]
        
        chan = rescale_intensity(img[label], out_range=(0, 1)) - af
        chan[chan < 0] = 0
        chan[roi == 0] = 0
        chans.append(chan)

        # chan = rescale_intensity(img[label].copy(), out_range=(0, 1))
        # chan[roi == 0] = 0
        # chans.append(chan)
        
    zonation_chans[label] = np.array(chans)

del label, chans, img, roi, chan, af

In [ ]:
def disp_chans(img, title=None, ncols=4):
    depth = len(img)
    nrows = depth // ncols if depth % ncols == 0 else depth // ncols + 1
    
    idx = 0
    fig, axes = plt.subplots(nrows, ncols, figsize=(3*ncols, 3.2*nrows))
    for r in range(nrows):
        for c in range(ncols):
            if idx >= depth:
                axes[r, c].axis('off')
                continue
            axes[r, c].imshow(img[idx])
            idx += 1
            
    plt.tight_layout()
    plt.suptitle(title, y=1.01)
    plt.show()
    return None

In [ ]:
def disp_optical_flow(z1, z2, title=None):
    # Reference: 
    # https://scikit-image.org/docs/stable/auto_examples/registration/plot_opticalflow.html
    v, u = optical_flow_ilk(z1, z2)
    norm = np.sqrt(u** + v**2)
    nvec = 20
    nl, nc = z1.shape
    step = max(nl//nvec, nc//nvec)
    
    y, x = np.mgrid[:nl:step, :nc:step]
    u_ = u[::step, ::step]
    v_ = v[::step, ::step]

    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(10, 3))
    ax1.imshow(z1)
    ax1.set_title('Slice z')

    ax2.imshow(z2)
    ax2.set_title('Slice z+1')
    
    ax3.imshow(norm)
    ax3.quiver(x, y, u_, v_, color='r', units='dots',
               angles='xy', scale=0.8, scale_units='xy', lw=2)
    ax3.set_title("Optical flow vector field")

    plt.tight_layout()
    plt.suptitle(title, y=1.1)
    plt.show()

In [ ]:
for label in zone_labels[:3]:
    print('Channel', label)
    print('==='*5)
    for i in range(len(zonation_chans[label])-1):
        z1, z2 = zonation_chans[label][i], zonation_chans[label][i+1]
        disp_optical_flow(z1, z2, title='Transition of {0} slices({1} vs. {2})'.format(label, i, i+1))   
    print('\n\n\n')

del z1, z2

#### 3D heat diffusion 
- Trajectory origin: `ASS` for now; destination `GS`
- 6-connected components as the graph unit


**(1)**. Construct 3D ROI & CV / PV tentative mask

**TODO**: expand # Z-slices & increase weights along in-plane (Z-axis) edges for anisotropy

In [ ]:
# CV / PV mask w/ otsu threshold

# roi = np.array(roi_masks)
# cv_mask = np.zeros_like(roi, dtype=np.int32)
# for i, chan in enumerate(zonation_chans['GS 647']):
#     threshold = threshold_otsu(chan)
#     cv_mask[i] = (chan > threshold).astype(np.int32)
# del chan, threshold

# pv_mask = np.zeros_like(roi, dtype=np.int32)
# for i, chan in enumerate(zonation_chans['ASS1 PE']):
#     threshold = threshold_otsu(chan)
#     pv_mask[i] = (chan > threshold).astype(np.int32)
# del chan, threshold

# mask = np.zeros_like(roi, dtype=np.int32)
# mask[np.logical_and(pv_mask == 1, cv_mask == 0)] = -1
# mask[np.logical_and(pv_mask == 0, cv_mask == 1)] = 1

# Dilate on mask
# for i, mask_slc in enumerate(mask):
#     cv_mask = binary_dilation(mask_slc == 1, np.ones((5, 5))).astype(np.uint8)
#     pv_mask = binary_dilation(mask_slc == -1, np.ones((5, 5))).astype(np.uint8)

#     mask[i][pv_mask == 1] = -1
#     mask[i][cv_mask == 1] = 1

# del cv_mask, pv_mask

# for i, (roi_slc, mask_slc) in enumerate(zip(roi, mask)):
#     fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))
#     ax1.imshow(roi_slc)
#     ax1.set_title('Tissue ROI (z={})'.format(i))
#     ax1.axis('off')
    
#     ax2.imshow(mask_slc, cmap='coolwarm')
#     ax2.set_title('CV/PV mask (z={})'.format(i))
#     ax2.axis('off')
    
#     plt.tight_layout()
#     plt.show()

# del roi_slc, mask_slc

# # Save current ROI & CV/PV mask
# tifffile.imwrite('../data/cycif/backup/CyIF_ROI.tif', roi)
# tifffile.imwrite('../data/cycif/backup/CyIF_zone_masks.tif', mask)

**(2)** Construct 3D graph & run diffusion inference

In [ ]:
# 3D heat diffusion model

# Load computed roi & mask
# DEBUG: 3D diffusion's correction w/ Z-slices 8-11
roi = tifffile.imread('../data/cycif/backup/CyIF_ROI.tif')[-4:]
mask = tifffile.imread('../data/cycif/backup/CyIF_zone_masks.tif')[-4:]


G = create_graph(roi=roi)
add_graph_props(G, cv_indices, pv_indices, depth=roi.shape[0])
u_i, interior_nodes = compute_interior_temp(G)
u = assign_diffusion_temp(u_i, 
                          interior_nodes,
                          cv_coords=np.where(mask == 1),
                          pv_coords=np.where(mask == -1),
                          shape=mask.shape)

In [ ]:
t0 = time.perf_counter()

model = zonation.HeatDiffusion(vein_prior=mask,
                               roi=roi,
                               ndim=3)
U_i, interior_nodes = model.get_interior_U()
U = model.infer_zone_dynamics()
# lobule_layers = model.infer_zones()

t1 = time.perf_counter()
print('Heat diffusion for dim {0} x {1} x {2} takes {2}s'.format(mask.shape[0], mask.shape[1], mask.shape[2], t1-t0))
del U_i, interior_nodes


In [ ]:
for i, u_slc in enumerate(U):
    plt.figure()
    plt.imshow(u_slc, cmap='coolwarm')
    plt.title('Diffused dynamics (z={})'.format(i))
    plt.show()

In [ ]:
# Save inferred heat dynamics
tifffile.imwrite('../data/cycif/backup/CyIF_zone_dynamics.ome.tif', u, metadata={'axes': 'ZYX'})

### Xenium

Dataset:
 - 2D DESI-paired xenium samples of 5-female + 5-male liver tissues
 - Vizgen MERFISH [Mouse liver](https://squidpy.readthedocs.io/en/stable/notebooks/tutorials/tutorial_vizgen_mouse_liver.html#hepatocyte-zonation)

Questions:
- Loading coords & expression matrices, check segmentation quality
- EDA: variance summary w/ PCA / pPCA
- Cell-type annotation & spatial distribution checks

In [ ]:
import IO, utils, plot
from importlib import reload
reload(IO)
reload(utils)
reload(plot)

In [ ]:
%ls ../data/xenium/
%ls ../data/xenium/output-XETG00169__0005109__Region_1__20240207__192326/

In [ ]:
data_path = '../data/xenium/'
sample_ids = sorted([f for f in os.listdir(data_path) if os.path.isdir(os.path.join(data_path, f))])

In [ ]:
zonal_markers = [
    'CYP2A7', 'CYP2B6',  # PP
    'CYP3A4',   # PC
]   

cell_type_markers = {
    'PP Hepatocytes': ['CYP2A7', 'CYP2B6'],
    'PC Hepatocytes': ['CYP3A4', 'ADH4'],
    'Cholangiocytes': ['KRT7', 'CFTR'],
    'Fibroblast': ['FBN1'],
    'Kupffer': ['CD14'],
    'SMC': ['MYH11', 'ACTA2'],
    'Endothelial': ['PECAM1', 'LYVE1'],
    'Lymphocytes': ['PTPRC', 'CD8A'],
    'Macrophages': ['CD163', 'MARCO'],
}

all_markers = list(dict.fromkeys(
    [marker for (_, markers) in cell_type_markers.items() 
     for marker in markers]
))

AGE_MAP = {
    'NIH_F1': 44, 'NIH_F2': 32, 'NIH_F3': 66, 'NIH_F4': 25, 'NIH_F5': 65,
    'NIH_M1': 34, 'NIH_M2': 28, 'NIH_M3': 47, 'NIH_M4': 63, 'NIH_M5': 30
}

WEIGHT_MAP = {
    'NIH_F1': 67.2, 'NIH_F2': 64.7, 'NIH_F3': 110.6, 'NIH_F4': 71.9, 'NIH_F5': 67.8,
    'NIH_M1': 79.5, 'NIH_M2': 70.5, 'NIH_M3': 82.8, 'NIH_M4': 153.7, 'NIH_M5': 102.9
}



In [ ]:
adata = sc.read_10x_h5(os.path.join(data_path, sample_ids[0], 'cell_feature_matrix.h5'))
with gzip.open(os.path.join(os.path.join(data_path, sample_ids[0], 'cells.csv.gz')), 'rt') as ifile:
    meta_df = pd.read_csv(ifile, index_col=[0])

adata.obs = meta_df.copy()
adata.obs['n_genes_by_counts'] = (adata.X > 0).sum(1).A.flatten()
adata.obsm['spatial'] = adata.obs[['x_centroid', 'y_centroid']].copy().to_numpy()
IO.load_spatial(adata, path=os.path.join(data_path, sample_ids[0], 'morphology_focus.ome.tif'), load_img=True)  # Append hi-res image

# QC
sc.pp.calculate_qc_metrics(adata, percent_top=(10, 20, 50, 150), inplace=True)
sc.pp.filter_cells(adata, min_counts=10)
sc.pp.filter_genes(adata, min_cells=5)

adata.layers['counts'] = adata.X.copy()
sc.pp.normalize_total(adata, inplace=True)
sc.pp.log1p(adata)
sc.tl.pca(adata, svd_solver="arpack")  # PCA as an alternative to scaling 
# sc.tl.diffmap(adata)
sc.pp.neighbors(adata, n_neighbors=40)
sc.tl.leiden(adata)
sc.tl.umap(adata)

adata.obs['Lognorm total counts'] = adata.X.A.sum(1)

# Store top PCs & DCs to adata.obs
n_features = 5
for i in range(n_features):
    adata.obs['PC'+str(i+1)] = adata.obsm['X_pca'][:, i]
    adata.obs['DC'+str(i+1)] = adata.obsm['X_diffmap'][:, i]

del meta_df, n_features

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(15, 4))

axs[0].set_title("Total transcripts per cell")
sns.histplot(
    adata.obs["total_counts"],
    kde=False,
    ax=axs[0],
)

axs[1].set_title("Unique transcripts per cell")
sns.histplot(
    adata.obs["n_genes_by_counts"],
    kde=False,
    ax=axs[1],
)

axs[2].set_title("Area of segmented cells")
sns.histplot(
    adata.obs["cell_area"],
    kde=False,
    ax=axs[2],
)

axs[3].set_title("Nucleus ratio")
sns.histplot(
    adata.obs["nucleus_area"] / adata.obs["cell_area"],
    kde=False,
    ax=axs[3],
)

del axs

Visualize PCA & clustering results:

In [ ]:
sc.pl.pca_variance_ratio(adata)

Known zonation / cell-type specific genes:

In [ ]:
sq.pl.spatial_scatter(adata, color=markers, size=20, img=False, cmap="Reds", ncols=3)

In [ ]:
# Moran's I: spatial autocorrelation
n_genes = 20
sq.gr.spatial_neighbors(adata, coord_type="generic", spatial_key="spatial")
sq.gr.spatial_autocorr(adata, mode="moran")

top_autocorr = (
    adata.uns["moranI"]["I"].sort_values(ascending=False).head(n_genes).index.tolist()
)
bot_autocorr = (
    adata.uns["moranI"]["I"].sort_values(ascending=True).head(n_genes).index.tolist()
)

sq.pl.spatial_scatter(
    adata, color=top_autocorr, size=20, cmap="Reds", img=False, figsize=(8, 5), ncols=3
)

sq.pl.spatial_scatter(
    adata, color=bot_autocorr, size=20, cmap="Reds", img=False, figsize=(8, 5), ncols=3
)

#### Cell-type annotation

Per-sample annotations

In [ ]:
sample_id = 'NIH_F2'

adata = IO.load_xenium(os.path.join(data_path, sample_id))
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)
sc.pp.pca(adata)
sc.pp.neighbors(adata, use_rep='X_pca')

sc.tl.umap(adata)
sc.tl.leiden(adata)
adata.obs.leiden = adata.obs.leiden.astype(np.uint8).astype('category')

In [ ]:
sc.pl.umap(adata, color='leiden')
sc.pl.umap(adata, color=all_markers, cmap='magma')

In [ ]:
cell_type_markers

In [ ]:
cluster_map = {
    'PP-Hep': [0,7],
    'PC-Hep': [1,6],
    'Cholangiocytes': [8],
    'Fibroblasts': [3],
    'Endothelial': [5,10],
    'Lymphocytes': [9],
    'Macrophages': [2],
    'Lymphocytes+SMC': [4],
}

for cell_type, cluster_id in cluster_map.items():
    adata.obs.leiden.replace(cluster_id, cell_type, inplace=True)

display(adata.obs.leiden.value_counts())

In [ ]:
pd.DataFrame(adata.obs.leiden).to_csv(os.path.join(data_path, sample_id, 'annots.csv'), 
                                      index=True)

**Annotate refined cell-types through subclusters**

#### All samples

In [ ]:
data_path = '../data/xenium/'
sample_ids = sorted([f for f in os.listdir(data_path) if os.path.isdir(os.path.join(data_path, f))])

In [ ]:
# Check batch effects w/ joint embedding
adatas = [sc.read_10x_h5(os.path.join(data_path, sample_id, 'cell_feature_matrix.h5')) 
          for sample_id in sample_ids]
adata_cat = sc.AnnData.concatenate(*adatas)
adata_cat.obs.batch = adata_cat.obs.batch.astype('category')

sc.pp.filter_cells(adata_cat, min_counts=10)
sc.pp.filter_genes(adata_cat, min_cells=5)
sc.pp.normalize_total(adata_cat, inplace=True)
sc.pp.log1p(adata_cat)
sc.tl.pca(adata_cat, svd_solver="arpack") 
sc.pp.neighbors(adata_cat)
sc.tl.diffmap(adata_cat)
sc.tl.umap(adata_cat)

# Store top PCs & DCs to adata.obs
n_features = 5
for i in range(n_features):
    adata_cat.obs['PC'+str(i+1)] = adata_cat.obsm['X_pca'][:, i]
    adata_cat.obs['DC'+str(i+1)] = adata_cat.obsm['X_diffmap'][:, i]

del adatas, n_features

In [ ]:
adata_cat.obs.head()

In [ ]:
sc.pl.umap(adata_cat, color='batch', frameon=False, title='Concatenated samples')

fig, axes = plt.subplots(2, 5, figsize=(20, 6))
idx = 0
batch_ids = sorted(adata_cat.obs.batch.unique())
for r in range(2):
    for c in range(5):
        ax = axes[r, c]
        ax = sc.pl.umap(adata_cat, color='batch', groups=batch_ids[idx], 
                        ax=ax, frameon=False, title='', show=False)
        idx += 1

plt.tight_layout()
plt.show()
del ax, r, c, idx, batch_ids, sample_id

In [ ]:
# Covariates: gender, age, weight (TODO: load metadata from .csv)
adata_cat.obs['gender'] = adata_cat.obs.batch.apply(lambda x: 'M' if 'M' in x else 'F')
adata_cat.obs['age'] = adata_cat.obs.batch.apply(lambda x: AGE_MAP[x]).astype(np.float32)
adata_cat.obs['weight'] = adata_cat.obs.batch.apply(lambda x: WEIGHT_MAP[x]).astype(np.float32)

sc.pl.umap(adata_cat, color=['gender', 'age', 'weight'], frameon=False, color_map='magma')

In [ ]:
cell_type_markers

In [ ]:
# Cell-type visualization
for cell_type, markers in cell_type_markers.items():
    print('Cell type: {}'.format(cell_type))
    sc.pl.umap(adata_cat, color=markers, frameon=False, color_map='magma')
    print('\n')

del cell_type, markers

In [ ]:
# TODO: Cell-type annotations

In [ ]:
# Save concatenated samples
adata_cat.write_h5ad(os.path.join(data_path, 'concat.h5ad'))

In [ ]:
# Check through individual samples: PCA, Leiden clustering, marker distributions & Spatially varying genes
import time

for sample_id in sample_ids:
    feature_path = os.path.join(data_path, sample_id, 'cell_feature_matrix.h5')
    meta_path = os.path.join(data_path, sample_id, 'cells.csv.gz')
    img_path = os.path.join(data_path, sample_id, 'morphology_focus.ome.tif')

    try:
        assert(os.path.isfile(feature_path) and os.path.isfile(meta_path) and os.path.isfile(img_path))
    except AssertionError:
        print("At least one of (1) feature matrix; (2). metadata; (3). paired spatial image doesn't exist for {}".format(sample_id))
        continue

    print("="*100)
    print("Loading {}...".format(sample_id))
    adata = sc.read_10x_h5(os.path.join(data_path, sample_id, 'cell_feature_matrix.h5'))
    with gzip.open(os.path.join(os.path.join(data_path, sample_id, 'cells.csv.gz')), 'rt') as ifile:
        meta_df = pd.read_csv(ifile, index_col=[0])
    
    adata.obs = meta_df.copy()
    adata.obs['n_genes_by_counts'] = (adata.X > 0).sum(1).A.flatten()
    adata.obsm['spatial'] = adata.obs[['x_centroid', 'y_centroid']].copy().to_numpy()
    IO.load_spatial(adata, path=os.path.join(data_path, sample_id, 'morphology_focus.ome.tif'), load_img=True)  # Append hi-res image
    
    # Standard preprocessing
    sc.pp.filter_cells(adata, min_counts=10)
    sc.pp.filter_genes(adata, min_cells=5)

    adata.layers['counts'] = adata.X.copy()
    sc.pp.normalize_total(adata, inplace=True)
    sc.pp.log1p(adata)
    sc.tl.pca(adata, svd_solver="arpack")  # PCA as an alternative to scaling 
    sc.pp.neighbors(adata)
    sc.tl.umap(adata)
    sc.tl.leiden(adata)

    # Spatial visualizations
    coords = adata.obs[['x_centroid', 'y_centroid']].copy().to_numpy()

    # (1). PCA explanation
    print("PCA:\n\n")
    sc.pl.pca_variance_ratio(adata)
    
    n_pcs = 4
    for i in range(n_pcs):
        col = 'PC'+str(i+1)
        adata.obs[col] = adata.obsm['X_pca'][:, i]
    sq.pl.spatial_scatter(adata, color=['PC'+str(i+1) for i in range(n_pcs)], size=20, img=False, cmap="Reds", ncols=4)
    del n_pcs, col

    # Leiden clustering
    # for i, label in enumerate(unique_labels):
    #     plt.scatter(*coords[adata.obs.leiden.to_numpy().astype(np.uint8) == label, :].T, s=3, c=colors[i], label='leiden {}'.format(label))
    # plt.title('Leiden', fontsize=20)
    # plt.legend(fontsize=10)
    # plt.show()

    # (2). Marker & spatially variable genes
    print("Spatial gene distributions:\n\n")
    markers = np.intersect1d(markers, adata.var_names)
    sq.pl.spatial_scatter(adata, color=markers, size=20, img=False, cmap="Reds", ncols=3)
    time.sleep(2.0)
    plt.show()

    n_genes = 18
    sq.gr.spatial_neighbors(adata, coord_type="generic", spatial_key="spatial")
    sq.gr.spatial_autocorr(adata, mode="moran")
    top_autocorr = (
        adata.uns["moranI"]["I"].sort_values(ascending=False).head(n_genes).index.tolist()
    )
    sq.pl.spatial_scatter(adata, color=top_autocorr, size=20, cmap="Reds", img=False, figsize=(8, 5), ncols=3)
    time.sleep(2.0)
    plt.show()
    
    print("="*100)
    print('\n\n') 

    del adata
    gc.collect()

del feature_path, meta_path, img_path

#### Trajectory Inference

Standard pseudotime inference (DPT, Palantir, scVelo (velocity pseudotime)):

In [ ]:
# # Individual sample
adata = sc.read_10x_h5(os.path.join(data_path, sample_ids[0], 'cell_feature_matrix.h5'))
with gzip.open(os.path.join(os.path.join(data_path, sample_ids[0], 'cells.csv.gz')), 'rt') as ifile:
    meta_df = pd.read_csv(ifile, index_col=[0])

adata.obs = meta_df.copy()
adata.obs['n_genes_by_counts'] = (adata.X > 0).sum(1).A.flatten()
adata.obsm['spatial'] = adata.obs[['x_centroid', 'y_centroid']].copy().to_numpy()
IO.load_spatial(adata, path=os.path.join(data_path, sample_ids[0], 'morphology_focus.ome.tif'), load_img=True)  # Append hi-res image

sc.pp.filter_cells(adata, min_counts=10)
sc.pp.filter_genes(adata, min_cells=5)
adata.raw = adata

adata.layers['counts'] = adata.X.copy()
sc.pp.normalize_total(adata, inplace=True)
sc.pp.log1p(adata)
sc.tl.pca(adata, svd_solver="arpack")  # PCA as an alternative to scaling 
sc.tl.diffmap(adata)
sc.pp.neighbors(adata, n_neighbors=40)
sc.tl.leiden(adata)
sc.tl.umap(adata)

adata.obs['Lognorm total counts'] = adata.X.A.sum(1)

# Store top PCs & DCs to adata.obs
n_features = 5
for i in range(n_features):
    adata.obs['PC'+str(i+1)] = adata.obsm['X_pca'][:, i]
    adata.obs['DC'+str(i+1)] = adata.obsm['X_diffmap'][:, i]

del meta_df, n_features

In [ ]:
sq.pl.spatial_scatter(adata, color='CYP1A1', size=10, img=False, cmap="Reds")

In [ ]:
# # Try root cell w/ CV-enriched cells
# dpt_marker = 'CYP1A1'
# adata.uns['iroot'] = np.argmax(adata[:, dpt_marker].X.A)
# sc.tl.dpt(adata, n_dcs=10, n_branchings=0)

# sq.pl.spatial_scatter(adata, color='dpt_pseudotime', size=10, img=False, cmap='turbo')
# fig, axes = plt.subplots(3, 1, figsize=(5, 5))
# sc.pl.scatter(adata, x='dpt_pseudotime', y='CYP1A1', ax=axes[0], show=False)
# sc.pl.scatter(adata, x='dpt_pseudotime', y='CYP3A4', ax=axes[1], show=False)
# sc.pl.scatter(adata, x='dpt_pseudotime', y='CYP2A7', ax=axes[2], show=False)
# plt.tight_layout()
# plt.show()

# # Try root cell w/ PV-enriched cells
# dpt_marker = 'DPT'
# adata.uns['iroot'] = np.argmax(adata[:, dpt_marker].X.A)
# sc.tl.dpt(adata, n_dcs=10, n_branchings=0)

# sq.pl.spatial_scatter(adata, color='dpt_pseudotime', size=10, img=False, cmap='turbo')
# fig, axes = plt.subplots(3, 1, figsize=(5, 5))
# sc.pl.scatter(adata, x='dpt_pseudotime', y='CYP1A1', ax=axes[0], show=False)
# sc.pl.scatter(adata, x='dpt_pseudotime', y='CYP3A4', ax=axes[1], show=False)
# sc.pl.scatter(adata, x='dpt_pseudotime', y='CYP2A7', ax=axes[2], show=False)
# plt.tight_layout()
# plt.show()

sq.pl.spatial_scatter(adata, color='dpt_pseudotime', size=10, img=False, cmap='turbo')
fig, axes = plt.subplots(3, 1, figsize=(5, 5))
sc.pl.scatter(adata, x='dpt_pseudotime', y='CYP1A1', ax=axes[0], show=False)
sc.pl.scatter(adata, x='dpt_pseudotime', y='CYP3A4', ax=axes[1], show=False)
sc.pl.scatter(adata, x='dpt_pseudotime', y='CYP2A7', ax=axes[2], show=False)
sc.pl.scatter(adata, x='dpt_pseudotime', y='DPT', ax=axes[2], show=False)
plt.tight_layout()
plt.show()

# Binned plots argsorted by "Trajectory"
idx = np.argsort(adata.obs.dpt_pseudotime).to_numpy()  
sorted_expr = adata.X.A[idx, :]
expr_means, expr_stds = utils.get_binned_features(sorted_expr, nbins=100)
expr_means = pd.DataFrame(expr_means, columns=adata.var_names)
expr_stds = pd.DataFrame(expr_stds, columns=adata.var_names)

plot.disp_desi_gradient(expr_means['CYP1A1'], expr_stds['CYP1A1'], vmax=np.max(expr_means))
plot.disp_desi_gradient(expr_means['CYP3A4'], expr_stds['CYP3A4'], vmax=np.max(expr_means))
plot.disp_desi_gradient(expr_means['CYP2A7'], expr_stds['CYP2A7'], vmax=np.max(expr_means))
plot.disp_desi_gradient(expr_means['DPT'], expr_stds['DPT'], vmax=np.max(expr_means))

del idx, sorted_expr, expr_means, expr_stds

In [ ]:
sq.pl.spatial_scatter(adata, color=['total_counts', 'Lognorm total counts', 'PC1'], size=10, img=False, cmap='turbo')

In [ ]:
sq.pl.spatial_scatter(adata, color='PC3', size=10, img=False, cmap='turbo')

fig, axes = plt.subplots(3, 1, figsize=(5, 5))
sc.pl.scatter(adata, x='PC3', y='CYP1A1', ax=axes[0], show=False)
sc.pl.scatter(adata, x='PC3', y='CYP3A4', ax=axes[1], show=False)
sc.pl.scatter(adata, x='PC3', y='CYP2A7', ax=axes[2], show=False)
sc.pl.scatter(adata, x='PC3', y='DPT', ax=axes[2], show=False)
plt.tight_layout()
plt.show()

# Binned plots argsorted by "Trajectory"
idx = np.argsort(adata.obs.PC3).to_numpy()  
sorted_expr = adata.X.A[idx, :]
expr_means, expr_stds = utils.get_binned_features(sorted_expr, nbins=100)
expr_means = pd.DataFrame(expr_means, columns=adata.var_names)
expr_stds = pd.DataFrame(expr_stds, columns=adata.var_names)

plot.disp_desi_gradient(expr_means['CYP1A1'], expr_stds['CYP1A1'], vmax=np.max(expr_means))
plot.disp_desi_gradient(expr_means['CYP3A4'], expr_stds['CYP3A4'], vmax=np.max(expr_means))
plot.disp_desi_gradient(expr_means['CYP2A7'], expr_stds['CYP2A7'], vmax=np.max(expr_means))
plot.disp_desi_gradient(expr_means['DPT'], expr_stds['DPT'], vmax=np.max(expr_means))

del idx, sorted_expr, expr_means, expr_stds

In [ ]:
sq.pl.spatial_scatter(adata, color='DC2', size=10, img=False, cmap='turbo')

fig, axes = plt.subplots(3, 1, figsize=(5, 5))
sc.pl.scatter(adata, x='DC2', y='CYP1A1', ax=axes[0], show=False)
sc.pl.scatter(adata, x='DC2', y='CYP3A4', ax=axes[1], show=False)
sc.pl.scatter(adata, x='DC2', y='CYP2A7', ax=axes[2], show=False)
sc.pl.scatter(adata, x='DC2', y='DPT', ax=axes[2], show=False)

plt.tight_layout()
plt.show()

# Binned plots argsorted by "Trajectory"
idx = np.argsort(adata.obs.DC2).to_numpy()  
sorted_expr = adata.X.A[idx, :]
expr_means, expr_stds = utils.get_binned_features(sorted_expr, nbins=100)
expr_means = pd.DataFrame(expr_means, columns=adata.var_names)
expr_stds = pd.DataFrame(expr_stds, columns=adata.var_names)

plot.disp_desi_gradient(expr_means['CYP1A1'], expr_stds['CYP1A1'], vmax=np.max(expr_means))
plot.disp_desi_gradient(expr_means['CYP3A4'], expr_stds['CYP3A4'], vmax=np.max(expr_means))
plot.disp_desi_gradient(expr_means['CYP2A7'], expr_stds['CYP2A7'], vmax=np.max(expr_means))
plot.disp_desi_gradient(expr_means['DPT'], expr_stds['DPT'], vmax=np.max(expr_means))

del idx, sorted_expr, expr_means, expr_stds

In [ ]:
# Concatenated samples

# Try root cell w/ CV-enriched cells
dpt_marker = 'CYP1A1'
adata_cat.uns['iroot'] = np.argmax(adata_cat[:, dpt_marker].X.A)
sc.tl.dpt(adata_cat, n_dcs=10, n_branchings=0)

fig, axes = plt.subplots(3, 1, figsize=(5, 5))
sc.pl.scatter(adata_cat, x='dpt_pseudotime', y='CYP1A1', ax=axes[0], show=False)
sc.pl.scatter(adata_cat, x='dpt_pseudotime', y='CYP3A4', ax=axes[1], show=False)
sc.pl.scatter(adata_cat, x='dpt_pseudotime', y='CYP2A7', ax=axes[2], show=False)
plt.tight_layout()
plt.show()

# Try root cell w/ CV-enriched cells
dpt_marker = 'DPT'
adata_cat.uns['iroot'] = np.argmax(adata_cat[:, dpt_marker].X.A)
sc.tl.dpt(adata_cat, n_dcs=10, n_branchings=0)

fig, axes = plt.subplots(3, 1, figsize=(5, 5))
sc.pl.scatter(adata_cat, x='dpt_pseudotime', y='CYP1A1', ax=axes[0], show=False)
sc.pl.scatter(adata_cat, x='dpt_pseudotime', y='CYP3A4', ax=axes[1], show=False)
sc.pl.scatter(adata_cat, x='dpt_pseudotime', y='CYP2A7', ax=axes[2], show=False)
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(5, 5))
sc.pl.scatter(adata_cat, x='PC2', y='CYP1A1', ax=axes[0], show=False)
sc.pl.scatter(adata_cat, x='PC2', y='CYP3A4', ax=axes[1], show=False)
sc.pl.scatter(adata_cat, x='PC2', y='CYP2A7', ax=axes[2], show=False)
sc.pl.scatter(adata_cat, x='PC2', y='DPT', ax=axes[2], show=False)
plt.tight_layout()
plt.show()

# Binned plots argsorted by "Trajectory"
idx = np.argsort(adata_cat.obs.PC2).to_numpy()  
sorted_expr = adata_cat.X.A[idx, :]
expr_means, expr_stds = utils.get_binned_features(sorted_expr, nbins=100)
expr_means = pd.DataFrame(expr_means, columns=adata.var_names)
expr_stds = pd.DataFrame(expr_stds, columns=adata.var_names)

plot.disp_desi_gradient(expr_means['CYP1A1'], expr_stds['CYP1A1'], vmax=np.max(expr_means))
plot.disp_desi_gradient(expr_means['CYP3A4'], expr_stds['CYP3A4'], vmax=np.max(expr_means))
plot.disp_desi_gradient(expr_means['CYP2A7'], expr_stds['CYP2A7'], vmax=np.max(expr_means))
plot.disp_desi_gradient(expr_means['DPT'], expr_stds['DPT'], vmax=np.max(expr_means))

del idx, sorted_expr, expr_means, expr_stds

#### Spatial clustering

Individual sample (`sample0`)

In [ ]:
# Individual sample
adata = sc.read_10x_h5(os.path.join(data_path, sample_ids[0], 'cell_feature_matrix.h5'))
with gzip.open(os.path.join(os.path.join(data_path, sample_ids[0], 'cells.csv.gz')), 'rt') as ifile:
    meta_df = pd.read_csv(ifile, index_col=[0])

adata.obs = meta_df.copy()
adata.obs['n_genes_by_counts'] = (adata.X > 0).sum(1).A.flatten()
adata.obsm['spatial'] = adata.obs[['x_centroid', 'y_centroid']].copy().to_numpy()
IO.load_spatial(adata, path=os.path.join(data_path, sample_ids[0], 'morphology_focus.ome.tif'), load_img=True)  # Append hi-res image

sc.pp.filter_cells(adata, min_counts=10)
sc.pp.filter_genes(adata, min_cells=5)
adata.raw = adata

adata.layers['counts'] = adata.X.copy()
sc.pp.normalize_total(adata, inplace=True)
sc.pp.log1p(adata)
sc.tl.pca(adata, svd_solver="arpack")  # PCA as an alternative to scaling 
sc.pp.neighbors(adata, use_rep='X_pca', n_neighbors=40)
sc.tl.diffmap(adata)
sc.tl.leiden(adata) 
sc.tl.umap(adata)

adata.obs['Lognorm total counts'] = adata.X.A.sum(1)
adata.obs.leiden = adata.obs.leiden.astype(np.uint8).astype('category')

# Store top PCs & DCs to adata.obs
n_features = 5
for i in range(n_features):
    adata.obs['PC'+str(i+1)] = adata.obsm['X_pca'][:, i]
    adata.obs['DC'+str(i+1)] = adata.obsm['X_diffmap'][:, i]

del meta_df, n_features

Leiden clustering & PAGA:

In [ ]:
sc.tl.paga(adata, groups='leiden')

In [ ]:
sq.pl.spatial_scatter(adata, color='leiden', size=10, img=False)

cluster_ids = sorted(adata.obs.leiden.unique())
ncol = 3
nrow = len(cluster_ids) // ncol if len(cluster_ids)%ncol == 0 \
    else len(cluster_ids) // ncol+1

fig, axes = plt.subplots(nrow, ncol, figsize=(5*ncol, 1.2*nrow))
idx = 0
for r in range(nrow):
    for c in range(ncol):
        if idx >= len(cluster_ids):
            axes[r, c].axis('off')
            continue
        ax = axes[r, c]
        ax = sq.pl.spatial_scatter(adata, color='leiden', groups=[idx], size=10, img=False,
                                   ax=ax, frameon=False, title='')
        idx += 1

plt.tight_layout()
plt.show()

del r, c, nrow, ncol, cluster_ids

stLearn:

In [ ]:
import stlearn as st

In [ ]:
# Try Louvain clustering
st.tl.clustering.louvain(adata, random_state=0)
sq.pl.spatial_scatter(adata, color='louvain', size=10, img=False)

cluster_ids = sorted(adata.obs.louvain.unique())
ncol = 3
nrow = len(cluster_ids) // ncol if len(cluster_ids)%ncol == 0 \
    else len(cluster_ids) // ncol+1

fig, axes = plt.subplots(nrow, ncol, figsize=(5*ncol, 1.2*nrow))
idx = 0
for r in range(nrow):
    for c in range(ncol):
        if idx >= len(cluster_ids):
            axes[r, c].axis('off')
            continue
        ax = axes[r, c]
        ax = sq.pl.spatial_scatter(adata, color='louvain', groups=[str(idx)], size=10, img=False,
                                   ax=ax, frameon=False, title='')
        idx += 1

plt.tight_layout()
plt.show()

del r, c, nrow, ncol, cluster_ids

In [ ]:
# PSeudo-Time-Space (PSTS) from Leiden clustering results
adata.uns['iroot'] = st.spatial.trajectory.set_root(
    adata,
    use_label='louvain',
    cluster='5',
    use_raw=True
)


In [ ]:
# "Global" trajectory btw clusters
adata.obs['imagerow'] = adata.obs['y_centroid'].copy()
adata.obs['imagecol'] = adata.obs['x_centroid'].copy()
st.spatial.trajectory.pseudotime(adata, eps=50, use_rep="X_pca", use_label="louvain")

In [ ]:
adata.obs['sub_cluster_labels'] = adata.obs['sub_cluster_labels'].astype(np.int16)
sq.pl.spatial_scatter(adata, color='sub_cluster_labels', cmap='turbo', groups=[0,1,2], size=10, img=False) 

In [ ]:
# Check stLearn 'subcluster', practically eqv. to binned trajectory
adata.obs['sub_cluster_labels'] = adata.obs['sub_cluster_labels'].astype(np.int16)
sq.pl.spatial_scatter(adata, color='sub_cluster_labels', cmap='turbo', size=10, img=False) 
adata.obs['sub_cluster_labels'] = adata.obs['sub_cluster_labels'].astype(str)

In [ ]:
# Dummy params.
adata.uns['spatial'][sample_ids[0]]['scalefactors'] = {'spot_diameter_fullres': 1.0,
                                                       'tissue_hires_scalef': 1.0}

st.spatial.trajectory.pseudotimespace_global(adata, use_label='louvain',
                                             list_clusters=['0','1','2','3','4','5','8'])

In [ ]:
G = st.utils._read_graph(adata, 'PTS_graph')
remove = [edge for edge in G.edges if 9999 in edge]
G.remove_edges_from(remove)
G.remove_node(9999)

import networkx as nx
nx.draw_networkx(G)
del remove

---